# XLM-RoBERTa Base multilingual Fake News experiment

This notebook is the GPU training path because local hardware is unsuitable: the installed PyTorch is CPU-only, the visible GTX 1650 has 4 GB VRAM, and only about 0.7 GB system RAM was available during the feasibility check. No local Transformer training was started.

## Colab setup
1. In Colab choose **Runtime > Change runtime type > GPU**. The first code cell stops if CUDA is unavailable.
2. Copy the project folder (or at least the six CSV split files with their directory structure) to Google Drive under `MyDrive/Community_Safety_Trust_AI/`.
3. Run cells from top to bottom. Training deliberately combines English and Hindi train/validation rows for one multilingual model. Test CSVs are not read until the selected checkpoint has been restored.

Labels are REAL=0 and FAKE=1. Original article text is passed to the multilingual tokenizer without translation. `max_length=256`. The checkpoint is selected by FAKE-class F1 on the concatenated validation set; per-language test reports are produced only after selection. The English/Hindi size imbalance means the combined validation F1 is sample-weighted toward English; per-language validation and test metrics are also reported.

In [ ]:
%pip -q install "transformers>=4.50,<5" "datasets>=2.18" "accelerate>=0.30" scikit-learn pandas matplotlib safetensors

In [ ]:
import os
import platform
import time
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before continuing; CPU training is intentionally disabled.')
for index in range(torch.cuda.device_count()):
    print('GPU', index, torch.cuda.get_device_name(index), round(torch.cuda.get_device_properties(index).total_memory / 1e9, 2), 'GB')
torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this if the project folder is stored elsewhere in Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/Community_Safety_Trust_AI')
SPLITS_DIR = PROJECT_ROOT / 'data' / 'fake_news' / 'splits'
MODEL_DIR = PROJECT_ROOT / 'models' / 'fake_news' / 'xlm_roberta' / 'selected'
CHECKPOINT_DIR = Path('/content/xlm_roberta_checkpoints')
RESULTS_DIR = PROJECT_ROOT / 'results' / 'fake_news' / 'xlm_roberta'
MODEL_NAME = 'FacebookAI/xlm-roberta-base'
MAX_LENGTH = 256
LABEL_NAMES = ['REAL', 'FAKE']

required_files = [SPLITS_DIR / f'{language}_{split}.csv' for language in ('english', 'hindi') for split in ('train', 'validation', 'test')]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError('Copy all split files into the Drive project folder first: ' + ', '.join(missing_files))
print('Project root:', PROJECT_ROOT)

## Dataset and leakage audit

A local diagnostic is saved as `results/fake_news/xlm_roberta/leakage_audit.md`. The split schema has no verified publisher, source, date, or topic ID. Publisher-name matches below are text mentions only. The existing random stratified split is not event- or time-disjoint, so topical/source shortcuts remain possible.

In [ ]:
audit_path = RESULTS_DIR / 'leakage_audit.json'
if audit_path.is_file():
    leakage_audit = json.loads(audit_path.read_text(encoding='utf-8'))
    for language, data in leakage_audit['languages'].items():
        print(language, 'exact duplicate overlap:', data['exact_duplicate_overlap'])
    print('Audit limitations:', *leakage_audit['limitations'], sep='\n- ')
else:
    print('Copy the local leakage_audit.json to the results directory to review source/topic cue counts.')

def read_split(language, split):
    path = SPLITS_DIR / f'{language}_{split}.csv'
    frame = pd.read_csv(path, encoding='utf-8-sig')
    if list(frame.columns) != ['text', 'label', 'language']:
        raise ValueError(f'Unexpected schema in {path}: {list(frame.columns)}')
    if frame.empty or frame.isna().any().any() or frame['text'].astype(str).str.strip().eq('').any():
        raise ValueError(f'Missing/blank rows in {path}')
    if not frame['language'].eq(language).all() or not frame['label'].isin([0, 1]).all():
        raise ValueError(f'Language or label mismatch in {path}')
    frame['label'] = frame['label'].astype('int64')
    return frame

# Only train and validation content is loaded before checkpoint selection.
train_frames = [read_split(language, 'train') for language in ('english', 'hindi')]
validation_frames = [read_split(language, 'validation') for language in ('english', 'hindi')]
train_df = pd.concat(train_frames, ignore_index=True)
validation_df = pd.concat(validation_frames, ignore_index=True)
print('Train rows by language/class:')
display(train_df.groupby(['language', 'label']).size().unstack(fill_value=0).rename(columns={0: 'REAL', 1: 'FAKE'}))
print('Validation rows by language/class:')
display(validation_df.groupby(['language', 'label']).size().unstack(fill_value=0).rename(columns={0: 'REAL', 1: 'FAKE'}))

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def to_tokenized_dataset(frame):
    dataset = Dataset.from_pandas(frame[['text', 'label']], preserve_index=False)
    def tokenize_batch(batch):
        return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
    dataset = dataset.map(tokenize_batch, batched=True, remove_columns=['text'])
    dataset.set_format(type='torch')
    return dataset

train_dataset = to_tokenized_dataset(train_df)
validation_dataset = to_tokenized_dataset(validation_df)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

In [ ]:
def compute_metrics(evaluation):
    logits = evaluation.predictions[0] if isinstance(evaluation.predictions, tuple) else evaluation.predictions
    predictions = np.asarray(logits).argmax(axis=-1)
    labels = np.asarray(evaluation.label_ids)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision_fake': precision_score(labels, predictions, pos_label=1, zero_division=0),
        'recall_fake': recall_score(labels, predictions, pos_label=1, zero_division=0),
        'f1_fake': f1_score(labels, predictions, pos_label=1, zero_division=0),
        'macro_f1': f1_score(labels, predictions, average='macro', zero_division=0),
    }

from transformers import AutoModelForSequenceClassification, EarlyStoppingCallback, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'REAL', 1: 'FAKE'},
    label2id={'REAL': 0, 'FAKE': 1},
)
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    overwrite_output_dir=True,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_fake',
    greater_is_better=True,
    gradient_checkpointing=True,
    fp16=True,
    optim='adamw_torch',
    logging_steps=100,
    report_to='none',
    dataloader_num_workers=2,
    seed=42,
    data_seed=42,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print('Configured FAKE-class validation F1 checkpoint selection. Test data remains unopened.')

In [ ]:
training_started = time.perf_counter()
train_output = trainer.train()
training_wall_seconds = time.perf_counter() - training_started
if trainer.state.best_model_checkpoint is None:
    raise RuntimeError('No best validation checkpoint was recorded.')
if trainer.state.best_metric is None:
    raise RuntimeError('Best validation FAKE F1 is missing.')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best validation FAKE F1:', trainer.state.best_metric)
print('Training wall time seconds:', training_wall_seconds)

## Separate final test evaluation

The best validation checkpoint is now restored in `trainer`. The next cell reads each test split once and reports each language independently. Do not use these test metrics to change settings or select another checkpoint.

In [ ]:
import matplotlib.pyplot as plt

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
test_reports = {}
for language in ('english', 'hindi'):
    test_frame = read_split(language, 'test')
    test_dataset = to_tokenized_dataset(test_frame)
    inference_started = time.perf_counter()
    prediction_output = trainer.predict(test_dataset, metric_key_prefix=f'{language}_test')
    inference_seconds = time.perf_counter() - inference_started
    logits = prediction_output.predictions[0] if isinstance(prediction_output.predictions, tuple) else prediction_output.predictions
    predicted_labels = np.asarray(logits).argmax(axis=-1)
    true_labels = test_frame['label'].to_numpy()
    report = classification_report(
        true_labels, predicted_labels, labels=[0, 1], target_names=LABEL_NAMES,
        output_dict=True, zero_division=0,
    )
    matrix = confusion_matrix(true_labels, predicted_labels, labels=[0, 1])
    result = {
        'language': language,
        'selected_checkpoint': trainer.state.best_model_checkpoint,
        'selection_metric': 'combined validation FAKE-class F1',
        'best_validation_fake_f1': float(trainer.state.best_metric),
        'test_rows': len(test_frame),
        'accuracy': float(accuracy_score(true_labels, predicted_labels)),
        'precision_fake': float(precision_score(true_labels, predicted_labels, pos_label=1, zero_division=0)),
        'recall_fake': float(recall_score(true_labels, predicted_labels, pos_label=1, zero_division=0)),
        'f1_fake': float(f1_score(true_labels, predicted_labels, pos_label=1, zero_division=0)),
        'macro_f1': float(f1_score(true_labels, predicted_labels, average='macro', zero_division=0)),
        'classification_report': report,
        'confusion_matrix_actual_rows_predicted_columns_REAL_FAKE': matrix.tolist(),
        'inference_seconds': inference_seconds,
        'inference_seconds_per_article': inference_seconds / len(test_frame),
    }
    test_reports[language] = result
    language_results = RESULTS_DIR / language
    language_results.mkdir(parents=True, exist_ok=True)
    (language_results / 'test_report.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    display_matrix = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=LABEL_NAMES)
    display_matrix.plot(cmap='Blues', values_format='d', colorbar=False)
    plt.title(f'{language.title()} test confusion matrix - XLM-RoBERTa Base')
    plt.tight_layout()
    plt.savefig(language_results / 'test_confusion_matrix.png', dpi=160)
    plt.close()
    print(language, json.dumps(result, ensure_ascii=False, indent=2))

run_metadata = {
    'model': MODEL_NAME,
    'label_mapping': {'REAL': 0, 'FAKE': 1},
    'training_languages': ['english', 'hindi'],
    'max_length': MAX_LENGTH,
    'learning_rate': 2e-5,
    'optimizer': 'AdamW (Transformers Trainer default)',
    'early_stopping_patience': 2,
    'selection_metric': 'combined validation FAKE-class F1',
    'best_checkpoint': trainer.state.best_model_checkpoint,
    'best_validation_fake_f1': float(trainer.state.best_metric),
    'training_wall_seconds': training_wall_seconds,
    'train_metrics': train_output.metrics,
    'test_reports': test_reports,
    'limitations': leakage_audit.get('limitations', []) if 'leakage_audit' in globals() else [],
}
(RESULTS_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved final model/tokenizer to:', MODEL_DIR)
print('Saved reports to:', RESULTS_DIR)

## Interpretation and limitations

This is a baseline fine-tuning experiment, not evidence that a claim is true or false. The source splits contain no verified publisher, publication date, or event/topic identifiers. Source-name and topic-keyword counts are heuristic text diagnostics; mentions may be quotations or article subjects rather than provenance. Random stratification does not prevent event-, source-, or time-related leakage. The Hindi training set is much smaller than English, so the combined validation objective is dominated by English examples. Review the separate per-language results and consider future publisher/event/time-grouped splits before drawing generalization conclusions.